# Workshop: ETL Pipeline Demo (raw_ecommerce_data.csv → warehouse.db)

## 1. EXTRACT — อ่านและสำรวจข้อมูลดิบ

In [1]:
import pandas as pd
import sqlite3

raw_df = pd.read_csv('raw_ecommerce_data.csv')

raw_df.info()
raw_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Order_ID       185 non-null    object
 1   Customer_Name  183 non-null    object
 2   Email          184 non-null    object
 3   Product        185 non-null    object
 4   Category       184 non-null    object
 5   Order_Date     185 non-null    object
 6   Quantity       185 non-null    int64 
 7   Unit_Price     185 non-null    object
 8   Amount         142 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB


,Order_ID,Customer_Name,Email,Product,Category,Order_Date,Quantity,Unit_Price,Amount
0,ORD-0036,Emma Brown,emma.brown@email.com,Monitor 24 inch,Electronics,24/04/2026,1,4131.00,"4,131.00"
1,ORD-0076,linda park,linda.park@email.com,Mechanical Keyboard,Electronics,"Apr 03, 2026",5,1669.50,8347.50
2,ORD-0101,john doe,john@email.com,Wireless Mouse,Electronics,11/04/2026,3,405.00,NaN
3,ORD-0059,Jane Smith,jane@email.com,Wireless Mouse,ELECTRONICS,13/03/2026,2,405.00,฿810.00
4,ORD-0038,PETER KIM,peter.kim@email.com,Gel Pen Set,Stationery,2026-02-28,5,85.50,427.50


## 2A. TRANSFORM — สร้าง dim_customer

แยก `Customer_Name` / `Email` ออกมาเป็น Dimension Table ก่อนจะ `drop_duplicates()` ต้องล้างช่องว่างและปรับตัวพิมพ์ให้เป็นมาตรฐานก่อน มิฉะนั้นชื่อเดียวกันจะถูกนับเป็นหลายแถวเพราะเว้นวรรค/ตัวพิมพ์ใหญ่-เล็ก (เช่น `'Alice Wong'`, `' Alice Wong '`, `'ALICE WONG'`)

In [2]:
raw_df['Customer_Name'] = raw_df['Customer_Name'].str.strip().str.title()
raw_df['Email'] = raw_df['Email'].str.strip().str.lower()

dim_customer = raw_df[['Customer_Name', 'Email']].drop_duplicates().reset_index(drop=True)
dim_customer['customer_id'] = dim_customer.index + 1
dim_customer = dim_customer[['customer_id', 'Customer_Name', 'Email']]

dim_customer

,customer_id,Customer_Name,Email
0,1,Emma Brown,emma.brown@email.com
1,2,Linda Park,linda.park@email.com
2,3,John Doe,john@email.com
3,4,Jane Smith,jane@email.com
4,5,Peter Kim,peter.kim@email.com
5,6,Michael Tan,michael.t@email.com
6,7,Nok Kwan,nok.kwan@email.com
7,8,Krit Som,krit.som@email.com
8,9,Thanawat Mee,thanawat.m@email.com
9,10,Ploy Kaew,ploy.kaew@email.com


## 2B. TRANSFORM — dim_product, dim_date และ fact_sales

ทำซ้ำกระบวนการเดียวกันกับมิติอื่น: `dim_product` (`Product`, `Category`) และ `dim_date` (แตก `Order_Date` เป็น ปี/เดือน/วัน) จากนั้นนำ Surrogate Key ของทุก Dimension กลับไป Merge เข้ากับตารางหลักเพื่อสร้าง `fact_sales`

`Unit_Price`/`Amount` ในไฟล์ดิบเก็บเป็นข้อความที่มีสัญลักษณ์เงิน (`฿`) และเครื่องหมายจุลภาคหลักพัน (`,`) ปนอยู่ ต้องแปลงเป็นตัวเลขก่อน และ `Amount` ที่ขาดหายจะคำนวณใหม่จาก `Quantity * Unit_Price`

In [3]:
def clean_currency(series):
    return (
        series.astype(str)
        .str.replace('฿', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace({'nan': None})
        .astype(float)
    )

raw_df['Unit_Price'] = clean_currency(raw_df['Unit_Price'])
raw_df['Amount'] = clean_currency(raw_df['Amount'])
raw_df['Amount'] = raw_df['Amount'].fillna(raw_df['Quantity'] * raw_df['Unit_Price'])

raw_df['Category'] = raw_df['Category'].str.strip().str.title()
raw_df['Product'] = raw_df['Product'].str.strip().str.title()

# dim_product
dim_product = raw_df[['Product', 'Category']].drop_duplicates().reset_index(drop=True)
dim_product['product_id'] = dim_product.index + 1
dim_product = dim_product[['product_id', 'Product', 'Category']]

# dim_date
raw_df['Date_Parsed'] = pd.to_datetime(raw_df['Order_Date'], dayfirst=True, format='mixed')
dim_date = raw_df[['Date_Parsed']].drop_duplicates().reset_index(drop=True)
dim_date['date_id'] = dim_date.index + 1
dim_date['Order_Date'] = dim_date['Date_Parsed'].dt.strftime('%Y-%m-%d')
dim_date['Year'] = dim_date['Date_Parsed'].dt.year
dim_date['Month'] = dim_date['Date_Parsed'].dt.month
dim_date['Day'] = dim_date['Date_Parsed'].dt.day
dim_date = dim_date[['date_id', 'Order_Date', 'Year', 'Month', 'Day']]

# fact_sales: ผูก Surrogate Key ของทุก Dimension กลับเข้าตารางหลัก
fact_sales = pd.merge(raw_df, dim_customer, on=['Customer_Name', 'Email'], how='left')
fact_sales = pd.merge(fact_sales, dim_product, on=['Product', 'Category'], how='left')
fact_sales['Order_Date_Format'] = fact_sales['Date_Parsed'].dt.strftime('%Y-%m-%d')
fact_sales = pd.merge(fact_sales, dim_date, left_on='Order_Date_Format', right_on='Order_Date', how='left')

fact_sales['sale_id'] = fact_sales.index + 1
fact_sales = fact_sales[[
    'sale_id', 'Order_ID', 'customer_id', 'product_id', 'date_id',
    'Quantity', 'Unit_Price', 'Amount'
]]

fact_sales.head()

,sale_id,Order_ID,customer_id,product_id,date_id,Quantity,Unit_Price,Amount
0,1,ORD-0036,1,1,1,1,4131.0,4131.0
1,2,ORD-0076,2,2,2,5,1669.5,8347.5
2,3,ORD-0101,3,3,3,3,405.0,1215.0
3,4,ORD-0059,4,3,4,2,405.0,810.0
4,5,ORD-0038,5,4,5,5,85.5,427.5


## 3A. LOAD — ตั้งค่า SQLite และสร้าง Dimension Tables

In [4]:
conn = sqlite3.connect('warehouse.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS fact_sales;")
cursor.execute("DROP TABLE IF EXISTS dim_customer;")
cursor.execute("DROP TABLE IF EXISTS dim_product;")
cursor.execute("DROP TABLE IF EXISTS dim_date;")

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_customer (
    customer_id INTEGER PRIMARY KEY,
    Customer_Name TEXT,
    Email TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_product (
    product_id INTEGER PRIMARY KEY,
    Product TEXT,
    Category TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY,
    Order_Date TEXT,
    Year INTEGER,
    Month INTEGER,
    Day INTEGER
)
''')

conn.commit()

## 3B. บังคับใช้ Relational Integrity (Foreign Keys)

In [5]:
cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute('''
CREATE TABLE IF NOT EXISTS fact_sales (
    sale_id INTEGER PRIMARY KEY,
    Order_ID TEXT,
    customer_id INTEGER,
    product_id INTEGER,
    date_id INTEGER,
    Quantity INTEGER,
    Unit_Price REAL,
    Amount REAL,
    FOREIGN KEY (customer_id) REFERENCES dim_customer(customer_id),
    FOREIGN KEY (product_id) REFERENCES dim_product(product_id),
    FOREIGN KEY (date_id) REFERENCES dim_date(date_id)
)
''')
conn.commit()

## 3C. ผลักข้อมูลเข้าสู่ SQLite

In [6]:
dim_customer.to_sql('dim_customer', conn, if_exists='append', index=False)
dim_product.to_sql('dim_product', conn, if_exists='append', index=False)
dim_date.to_sql('dim_date', conn, if_exists='append', index=False)
fact_sales.to_sql('fact_sales', conn, if_exists='append', index=False)

print("ETL Pipeline Completed Successfully!")

ETL Pipeline Completed Successfully!


## 4. Verification — ทดสอบคิวรีจาก Warehouse

In [7]:
test_query = '''
SELECT
    c.Customer_Name,
    SUM(f.Amount) AS Total_Spend
FROM fact_sales f
JOIN dim_customer c ON f.customer_id = c.customer_id
GROUP BY c.customer_id
ORDER BY Total_Spend DESC
LIMIT 3;
'''
result = pd.read_sql_query(test_query, conn)
print(result)

conn.close()
result

  Customer_Name  Total_Spend
0     Peter Kim     96283.25
1      Krit Som     88853.00
2    Alice Wong     81977.50


,Customer_Name,Total_Spend
0,Peter Kim,96283.25
1,Krit Som,88853.00
2,Alice Wong,81977.50
